# Phase S1 Debug: 4 configs, 50 epochs, verbose output

In [ ]:
import sys, os, json, math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch import linalg
import warnings
warnings.filterwarnings('ignore')

# Force unbuffered output
sys.stdout.reconfigure(line_buffering=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"STEP 1: Device = {device}", flush=True)

os.chdir('ThermoRG-NN')
print(f"STEP 2: CWD = {os.getcwd()}", flush=True)
print(f"Files: {os.listdir('.')[:5]}", flush=True)

In [ ]:
# Config: 4 runs
CONFIGS = [
    ('None_NoSkip',  'none',       False),
    ('LN_NoSkip',    'layernorm',  False),
    ('BN_NoSkip',    'batchnorm',  False),
    ('None_Skip',    'none',       True),
]

D_VALUES = [64]
SEEDS = [42]
EPOCHS = 50
BATCH_SIZE = 128
LR = 0.01
MOMENTUM = 0.9

OUT_DIR = Path('./phase_s1_results')
OUT_DIR.mkdir(exist_ok=True)

print(f"STEP 3: Configs = {len(CONFIGS)}, Runs = {len(CONFIGS)*len(D_VALUES)*len(SEEDS)}", flush=True)

In [ ]:
# Network
class Net(nn.Module):
    def __init__(self, base_ch=64, norm='none', skip=False):
        super().__init__()
        self.b1 = nn.Conv2d(3, base_ch, 3, padding=1, bias=False)
        self.b2 = nn.Conv2d(base_ch, base_ch*2, 3, padding=1, bias=False)
        self.b3 = nn.Conv2d(base_ch*2, base_ch*2, 3, padding=1, bias=False)
        if norm == 'layernorm':
            self.n1 = nn.LayerNorm([base_ch, 32, 32])
            self.n2 = nn.LayerNorm([base_ch*2, 32, 32])
            self.n3 = nn.LayerNorm([base_ch*2, 32, 32])
        elif norm == 'batchnorm':
            self.n1 = nn.BatchNorm2d(base_ch)
            self.n2 = nn.BatchNorm2d(base_ch*2)
            self.n3 = nn.BatchNorm2d(base_ch*2)
        else:
            self.n1 = self.n2 = self.n3 = nn.Identity()
        self.act = nn.GELU()
        if skip:
            self.s1 = nn.Identity()
            self.s2 = nn.Conv2d(base_ch, base_ch*2, 1) if base_ch != base_ch*2 else nn.Identity()
        else:
            self.s1 = self.s2 = None
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(base_ch*2, 10)

    def forward(self, x):
        x = self.act(self.n1(self.b1(x)))
        if self.s1: x = x + self.s1(x)
        x = self.act(self.n2(self.b2(x)))
        if self.s2: x = x + self.s2(x)
        x = self.act(self.n3(self.b3(x)))
        return self.fc(self.pool(x).squeeze())

print("STEP 4: Network class defined", flush=True)

In [ ]:
# J_topo
def compute_J_topo(net):
    ws = [m.weight.data for m in net.modules() if isinstance(m, nn.Conv2d)]
    eta_vals, d_prev = [], 3.0
    for w in ws:
        wf = w.view(w.size(0), -1)
        fro_sq = (wf**2).sum().item()
        S = linalg.svd(wf.to('cpu')).S
        spec_sq = S[0].item()**2 + 1e-12
        De = fro_sq / spec_sq
        eta = De / max(d_prev, 1e-8)
        eta_vals.append(max(eta, 1e-8))
        d_prev = De
    L = len(eta_vals)
    log_sum = sum(abs(math.log(max(e, 1e-12))) for e in eta_vals)
    return math.exp(-log_sum / L) if L > 0 else 0.0

print("STEP 5: J_topo function defined", flush=True)

In [ ]:
# Data
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])
transform_val = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

train_ds = CIFAR10('./data', train=True, transform=transform_train, download=True)
val_ds   = CIFAR10('./data', train=False, transform=transform_val, download=False)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"STEP 6: Data loaded - train={len(train_ds)}, val={len(val_ds)}", flush=True)

In [ ]:
# Training
def train_model(net, epochs, lr, momentum):
    net = net.to(device)
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=momentum, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    print(f"    Starting training...", flush=True)
    for ep in range(epochs):
        net.train()
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            crit(net(X), y).backward()
            opt.step()
        sched.step()
        if (ep+1) % 10 == 0:
            net.eval()
            loss = 0
            with torch.no_grad():
                for X, y in val_dl:
                    loss += crit(net(X.to(device)), y.to(device)).item() * X.size(0)
            print(f"    Epoch {ep+1}/{epochs}: val_loss={loss/len(val_ds):.4f}", flush=True)
    
    # Final eval
    net.eval()
    final_loss = 0
    with torch.no_grad():
        for X, y in val_dl:
            final_loss += crit(net(X.to(device)), y.to(device)).item() * X.size(0)
    return final_loss / len(val_ds)

print("STEP 7: Training function defined", flush=True)

In [ ]:
# MAIN LOOP
from datetime import datetime

results = {
    'timestamp': datetime.now().isoformat(),
    'epochs': EPOCHS,
    'configs': []
}

total_start = time.time()
run_count = 0

print(f"\n{'='*60}", flush=True)
print(f"MAIN LOOP STARTING - {len(CONFIGS)} configs", flush=True)
print(f"{'='*60}\n", flush=True)

for cfg_name, norm_type, use_skip in CONFIGS:
    print(f"\n{'='*50}", flush=True)
    print(f"CONFIG: [{cfg_name}] norm={norm_type}, skip={use_skip}", flush=True)
    print(f"{'='*50}", flush=True)
    
    cfg_start = time.time()
    cfg_result = {'name': cfg_name, 'norm': norm_type, 'skip': use_skip}
    
    # J_topo at init
    print(f"  Computing J_topo...", flush=True)
    model_init = Net(64, norm_type, use_skip)
    J_topo_init = compute_J_topo(model_init)
    cfg_result['J_topo_init'] = J_topo_init
    print(f"  J_topo(init) = {J_topo_init:.4f}", flush=True)
    del model_init
    torch.cuda.empty_cache()
    
    # Train for each D value
    for base_ch in D_VALUES:
        for seed in SEEDS:
            run_count += 1
            print(f"\n  --- Run {run_count}/{len(CONFIGS)*len(D_VALUES)*len(SEEDS)} ---", flush=True)
            print(f"  base_ch={base_ch}, seed={seed}", flush=True)
            
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            
            model = Net(base_ch, norm_type, use_skip)
            run_start = time.time()
            
            val_loss = train_model(model, EPOCHS, LR, MOMENTUM)
            
            run_time = (time.time() - run_start) / 60
            
            print(f"  Finished! val_loss={val_loss:.4f} in {run_time:.1f} min", flush=True)
            
            cfg_result[f'ch{base_ch}_s{seed}'] = {
                'val_loss': val_loss,
                'time_min': run_time
            }
            
            del model
            torch.cuda.empty_cache()

    cfg_time = (time.time() - cfg_start) / 60
    cfg_result['total_time_min'] = cfg_time
    results['configs'].append(cfg_result)
    
    print(f"\n  [{cfg_name}] completed in {cfg_time:.1f} min", flush=True)

total_time = (time.time() - total_start) / 60
results['total_time_min'] = total_time
print(f"\n{'='*60}", flush=True)
print(f"ALL DONE! Total time: {total_time:.1f} min", flush=True)
print(f"{'='*60}", flush=True)

In [ ]:
# Save
out_file = OUT_DIR / 'phase_s1_debug.json'
with open(out_file, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nSaved to {out_file}", flush=True)

In [ ]:
# Summary
print("\n" + "="*60, flush=True)
print("SUMMARY", flush=True)
print("="*60, flush=True)
for cfg in results['configs']:
    losses = [v['val_loss'] for k, v in cfg.items() if k.startswith('ch')]
    avg_loss = sum(losses) / len(losses) if losses else 0
    print(f"{cfg['name']:<15} J={cfg['J_topo_init']:.4f}  avg_loss={avg_loss:.4f}", flush=True)
print("\nDONE!", flush=True)